# Problem: fields

Two spatial fields reconstructed from readings at a few places: monthly-mean
air temperature over the Alps from NOAA stations, and a simulated radio
field in a building. The question is the project's usual one on a map
instead of a curve: what does a physical law buy the reconstruction, and
what does it cost where the law is incomplete?

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from physprior.viz import plots as P
from physprior.io import load_table, load_json
from physprior.config import get_settings
from physprior.benchmark.protocol import fit_arm

RESULTS = get_settings().results_dir
P.use_style()
pd.set_option("display.width", 160, "display.max_columns", 60)

def gif(path, width=560, caption=""):
    "Show an animation from figures/ without embedding it in the notebook."
    cap = f"<div style='color:#52514e;font-size:90%'>{caption}</div>" if caption else ""
    return HTML(f"<img src='../figures/{path}' width='{width}'>{cap}")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


## 1 · The problem — two laws for two maps

**Temperature.** Near the surface, a monthly mean falls with height at a
nearly constant lapse rate $\Gamma$, with a weak horizontal gradient:

$$T(\lambda, \phi, z) = T_0 + a\,(\lambda - \lambda_0)
  + b\,(\phi - \phi_0) + \Gamma z$$

The standard atmosphere gives $\Gamma = -6.5$ K/km. The test is extrapolation
up the mountain: fit the lower stations, predict the highest ones.

**Radio.** A transmitter's received power falls with distance as

$$P(d) = P_0 - 10\,n \log_{10}(d / d_0)$$

with the transmitter position unknown, so fitting the law also locates the
transmitter. In free space the law is complete; walls make it incomplete.

## 2 · The data — pulled and created

**Pulled**: NOAA ISD-Lite hourly station records for 2023, reduced to
monthly means of the 12 UTC observation by a rule fixed before any fit
(`physprior.data.sources.isd.SnapshotRule`).

**Created**: the radio field, solved from the 2-D Helmholtz equation with an
absorbing boundary, and a simulated temperature control at the real station
positions with a known lapse rate.

In [2]:
st = pd.read_csv(RESULTS / "fields/weather/july/stations.csv")
print(f"{len(st)} stations, {st.elev_m.min() / 1e3:.2f} to "
      f"{st.elev_m.max() / 1e3:.2f} km above sea level")
display(gif("fields/weather_data.png", 620))
display(gif("fields/rf_scene.png", 620))

197 stations, 0.00 to 3.58 km above sea level


## 3 · The method — five arms, plus kriging

## 3 · The method

Two different jobs travel under the name *physics-informed*, and this track does the following:

1. **Discovery** — find a law nobody supplied. Only `sr` (symbolic regression) can do this; it is given numbers and an operator set, never an equation.

2. **Recovery** — measure a constant *inside* a law that is supplied. `physics` fits it classically with a covariance; `pinn` fits it with a neural correction alongside, and `w_phys` sets how much correction is allowed.

**Arms run here:** oracle, physics, pinn, sr, nn

Kriging (ordinary, and on the law's residuals) is reported beside the arms, not as one. The radio track has no `sr` arm.

## 4 · The results — what the numbers say

The recovered lapse rate, and each arm's error at the highest stations:

In [3]:
ex = pd.read_csv(RESULTS / "fields/weather/july/extrapolation.csv")
display(ex.pivot_table(index="arm", columns="seed", values="nrmse_out").round(3))
display(pd.read_csv(RESULTS / "fields/weather/july/kriging.csv").round(3))
display(gif("fields/weather_extrapolation_july.png", 620))

seed,11,23,42
arm,,,
nn,1.117,0.914,0.591
oracle,0.265,0.265,0.265
physics,0.374,0.374,0.374
pinn,0.375,0.374,0.375
pinn_static,3.434,3.438,3.434
sr,0.356,0.356,0.356


,track,arm,sweep,rmse_in,nrmse_in,mape_in,rmse_out,nrmse_out,mape_out,seconds,n_free,expression,interpretable,dTdz,hyper_l_h_km,hyper_l_z_km,hyper_s_f,hyper_s_n,converged,starts_agreeing,param_T0,param_a,param_b,param_Gamma,err_Gamma_pct,sigma_T0,sigma_a,sigma_b,sigma_Gamma,block
0,fields/weather/july,gp,elevation,0.787,0.149,2.548,6.877,1.298,75.534,2.307,5,NaN,False,-5.108,1000.000,1.826,12.947,0.813,False,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fields/weather/july,uk,elevation,0.349,0.066,1.103,1.241,0.234,11.664,2.028,8,NaN,True,-7.118,34.973,0.544,0.965,0.472,True,3,28.032,0.121,-1.217,-6.565,1.008,0.306,0.082,0.268,0.383,NaN
2,fields/weather/july,gp,block,0.630,0.119,2.735,3.395,0.641,13.771,2.095,5,NaN,False,-6.728,212.345,3.375,16.958,0.697,True,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,fields/weather/july,uk,block,0.553,0.104,2.435,1.109,0.209,4.003,1.216,8,NaN,True,-7.459,46.540,10.000,1.098,0.630,False,1,28.774,-0.008,-1.006,-7.453,14.660,0.374,0.162,0.390,0.099,0.0
4,fields/weather/july,gp,block,0.916,0.173,3.840,1.011,0.191,5.671,1.813,5,NaN,False,-6.467,1000.000,3.744,16.093,0.948,False,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
5,fields/weather/july,uk,block,0.467,0.088,1.792,0.922,0.174,4.307,1.838,8,NaN,True,-7.451,39.227,2.186,0.956,0.577,True,3,28.441,0.123,-1.055,-7.338,12.886,0.260,0.089,0.283,0.145,1.0
6,fields/weather/july,gp,block,0.852,0.161,3.566,1.126,0.213,5.171,2.823,5,NaN,False,-6.520,581.895,3.166,17.999,0.897,True,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
7,fields/weather/july,uk,block,0.313,0.059,1.226,0.746,0.141,3.270,1.527,8,NaN,True,-7.389,36.938,0.505,1.134,0.459,True,2,28.313,0.144,-1.396,-7.005,7.775,0.320,0.083,0.308,0.193,2.0
8,fields/weather/july,gp,block,0.850,0.160,3.577,1.418,0.268,6.486,2.632,5,NaN,False,-6.553,1000.000,3.880,20.000,0.878,False,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
9,fields/weather/july,uk,block,0.337,0.064,1.312,1.299,0.245,5.208,1.757,8,NaN,True,-7.512,34.342,0.485,0.948,0.465,True,2,28.468,0.212,-1.718,-7.197,10.717,0.411,0.151,0.304,0.187,3.0


The radio field, with walls: each arm's reconstruction and the recovered
transmitter position.

In [4]:
display(gif("fields/rf_reconstruction_walls.png", 620))
display(gif("fields/rf_tx_recovery.png", 620))

The conclusion below covers the July temperature track. The radio track
keeps both scenes in one results file, and a verdict pooled across a scene
where the law is complete and one where it is not would mean nothing; its
per-scene tables are in `docs/fields/rf.md`.

---

## Conclusion

Everything below is rendered from `results/` at execution time by
`physprior.reporting.conclusions` — the verdicts, the margins and the prose.
A gap smaller than the seed-to-seed spread on the reporting seeds
(11 / 23 / 42) is reported as a **tie**, not rounded into a win.

In [5]:
from physprior.reporting import conclusions as C
verdicts = C.verdict_frame('fields')
display(verdicts)

,track,question,winner,runner-up,margin,seed spread,verdict
0,fields/weather/july,interpolation,nn,physics,1.01x,0.033,TIE (within seed spread)
1,fields/weather/july,data efficiency,physics,pinn,1x,0.016,TIE (within seed spread)
2,fields/weather/july,noise,nn,pinn,1.14x,0.028,TIE (within seed spread)
3,fields/weather/july,extrapolation,sr,physics,1.05x,8.7e-05,WIN
4,fields/weather/july,parameter recovery (Gamma),pinn,physics,1.48x,3.5,TIE (within seed spread)


In [6]:
from IPython.display import Markdown
display(Markdown(C.conclusion_markdown('fields')))

Across 5 protocol questions on 1 track(s), **1 produced a decisive answer** and 4 were inside the seed-to-seed spread and are reported as ties rather than rounded into wins. **`sr` takes 1 of the decisive question(s)**. The largest single gap is on **fields/weather/july / extrapolation**, where `sr` beats `physics` by **1.05x**. The `pinn` arm — the one this project is built around — wins **0**, loses **1** and ties **4** of these, which is the number to read before any claim about what a physics prior buys.

In [7]:
lines = ["- " + line for line in C.what_would_change_this('fields')]
display(Markdown("**What would change this conclusion**\n\n" + "\n".join(lines)))

**What would change this conclusion**

- `sr` wins 1 question(s) (july extrapolation): symbolic regression's reach is set by its operator set, which is a prior. Adding or removing an operator changes what it can find at all, so these wins are contingent on a choice made in the track definition.
- 4 question(s) are ties: more seeds, or a lower-variance fit (ensembling, early stopping), would decide them either way.
- The 1.05x gap on fields/weather/july / extrapolation rests on three seeds with a spread of 8.7e-05; it would be overturned by a seed set on which `physics` closes to within that spread, which is why the margin and the spread are printed side by side above.